# Investigate the incident. Watch the model bill.
### LangChain × NVIDIA NeMo Switchyard

One agent. The same evidence. Three ways to choose a model.

**Capable only** → the chosen stronger model handles every step.  
**Efficient only** → a smaller model handles every step.  
**Switchyard** → NVIDIA's router chooses between them.

This copy is configured for **live endpoint calls**. Check `MODE` and `ALLOW_PAID_CALLS`
before running cells. Set `MODE = "rehearsal"` for a local scripted rehearsal with no paid calls.

**Presenter:** spend 5–7 minutes on sections 2–5. Run the evaluation before the event and keep a recorded result ready.

## 1 · Set up once

Open a terminal in this folder, create a Python 3.12 environment, and install the pinned requirements:

```sh
uv venv --python 3.12
uv pip install --python .venv/bin/python -r requirements.txt
.venv/bin/python -m jupyterlab incident_routing_demo.ipynb
```

The pins matter: the current LangChain integration uses Switchyard 0.2.0's API. Upgrading
Switchyard alone to 0.3.0 breaks it. All routing below uses the real library.

The application is a small LangChain tool-using agent. It has no infrastructure to provision and
only reads the supplied synthetic data. The endpoint client uses OpenAI-compatible Chat Completions.

In [ ]:
from pathlib import Path
from getpass import getpass
import os
import json
import importlib.metadata

from dotenv import load_dotenv
from IPython.display import display, HTML
from incident_demo import (
    ROOT, load_settings, configuration_table, make_router, dataset_table, load_data, run_case, run_comparison,
    show_case, show_summary, progress_printer, save_results, load_results,
)

load_dotenv(ROOT / ".env", override=True)
assert importlib.metadata.version("nemo-switchyard") == "0.2.0", "Install requirements.txt in a clean Python 3.12 environment."
print("Switchyard", importlib.metadata.version("nemo-switchyard"), "· LangChain", importlib.metadata.version("langchain"))

### Choose a model profile and routing strategy

Model configuration follows [your learning-labs profile format](https://github.com/siddBanPsu/nvidia-ai-learning-labs/blob/main/configs/model-profiles/nemotron-lightning-ultra.toml):

- **Model profile TOML:** shared clients and the `efficient`, `capable`, and `classifier` roles.
- **Routing TOML:** Switchyard strategy and thresholds, plus run limits.
- **`.env`:** just the shared endpoint URL and API key for the selected client.
- **Optional pricing TOML:** verified rates once per client/model; classifier and efficient roles reuse them automatically.

The selected profile uses **Lightning as executor and classifier, and Claude Opus as capable
executor**, through the configured NVIDIA gateway. The other profile uses Lightning + Ultra.
Confirm model IDs and access on your endpoint. “Capable only” means the selected stronger model.

Both executors need tool calling; the classifier needs JSON-schema output. Missing prices or
usage produce **unavailable** costs, never zero. Free API credits do not demonstrate savings.
**Live cells spend money each time they run.** Start small; limits are in the strategy file.

In [ ]:
MODE = "rehearsal"          # "live" for real endpoint calls, "rehearsal" for local simulation
ALLOW_PAID_CALLS = False    # set True intentionally when ready

# PROFILE = ROOT / "configs/model-profiles/nemotron-lightning-ultra.toml"
PROFILE = ROOT / "configs/model-profiles/nemotron-nano-claude-opus.toml"
STRATEGY = ROOT / "configs/routing/task-classifier.toml"  # or "stage.toml"
PRICING = ROOT / "configs/pricing.toml"                  # optional; may be None

settings = load_settings(PROFILE, STRATEGY, PRICING, mode=MODE,
                         allow_paid_calls=ALLOW_PAID_CALLS)

if MODE == "live" and ALLOW_PAID_CALLS:
    roles = ("efficient", "frontier", "judge") if settings.routing == "task_classifier" else ("efficient", "frontier")
    for role in roles:
        ep = getattr(settings, role)
        if not os.getenv(ep.key_env):
            os.environ[ep.key_env] = getpass(f"{ep.key_env}: ")

settings.validate()
print(settings.profile_name)
print("Effective routing:", "stage router + scripted models" if MODE == "rehearsal" else settings.routing)
display(configuration_table(settings))

### Inspect the actual configuration

The next cell shows the files in use. Change the TOML, then rerun the configuration cell above.
This notebook's loader maps the routing preset to the real Switchyard Python constructors.
The routing preset is **adapter configuration**, not native `switchyard-server --config` TOML.
The shared-client model profile format matches your reference; this notebook adapter supports
`openai_chat` endpoints and rejects unsupported formats explicitly.

In [ ]:
from IPython.display import Code
display(Code(PROFILE.read_text(), language="toml"))
display(Code(STRATEGY.read_text(), language="toml"))

# Show exactly where these settings enter NVIDIA Switchyard.
import inspect
display(Code(inspect.getsource(make_router), language="python"))

### What the router does

In **live task-classifier mode**, Switchyard asks a judge to assess the current task/history and
selects the efficient or frontier model. Every judge call appears in the cost ledger. No
incident answer key or difficulty label is passed to the live agent or router.

In **stage mode**, Switchyard uses tool-activity signals and makes no judge calls. Those
heuristics may select the efficient model throughout this workload; switching is not guaranteed.
Rehearsal uses this real stage router with fixed, answer-key-driven model scripts.

The published LangChain 74% result used a different, judge-after-response escalation strategy.
This notebook measures its own configuration and makes no assumption about savings.

## 2 · Meet the incident

**Presenter, 0:00–0:45:** “An error message is a symptom. Can our agent gather enough evidence
to propose the right fix—and how much model spend does that require?”

Choose `INC-001` for an expired credential, or `INC-004` for a valid token rejected by a worker
whose clock is one hour ahead. The latter demonstrates why reading just one error log is insufficient.

In [ ]:
DEMO_CASE = "INC-004"
cases, runbooks, _ = load_data()
case = next(c for c in cases if c["id"] == DEMO_CASE)
display(HTML("<h3>Support ticket</h3>"))
print(json.dumps(case["ticket"], indent=2))
display(dataset_table().query("split == 'development'").reset_index(drop=True))

## 3 · Investigate with Switchyard

**Presenter, 0:45–2:30:** Watch the agent read the ticket, search logs, inspect metrics, consult
a runbook, and submit a proposed resolution. This environment has no tool that executes changes.

Rehearsal follows a known script. Live mode uses actual endpoint responses. Allow up to three
minutes; use recorded live results if the endpoint is too slow for the stage.

In [ ]:
demo_result = await run_case(DEMO_CASE, "switchyard", settings, on_event=progress_printer)
show_case(demo_result)

In [ ]:
print("Router error:", demo_result["error_type"])

for call in demo_result["calls"]:
    if call["status"] == "error":
        print(call["role"], call["model"], call.get("error_type"))

### Inspect the evidence and routing decisions

**Presenter, 2:30–3:30:** “Here is what the agent actually read, which model handled each turn,
and the accumulated model cost.” Executor and judge calls both count. A missing total means
the accounting is incomplete, not that the run was free.

In [ ]:
for event in demo_result["tool_events"]:
    if event["tool"] != "submit_report":
        print(f"\n{event['tool']} {event['arguments']}")
        print(json.dumps(event["response"], indent=2))

print("\nActual Switchyard routing metadata:")
print(json.dumps(demo_result["routing_trace"], indent=2))

## 4 · Compare the same tasks three ways

**Presenter, 3:30–5:15:** Use a comparison computed before the talk. Each arm uses the same
prompt, evidence, tools, and output cap. Case/arm order is shuffled; each task gets fresh state.
The comparison covers two cases (six agent tasks) when enabled. Check `RUN_COMPARISON` below
before running: this copy enables the comparison in live mode. Choose all six development cases once live calls work.

Cost per pass divides **the cost of all attempted tasks**, including failures, by the number
that pass. The pass rubric checks cause/action codes, decisive evidence actually retrieved,
the correct runbook, and a report with verification steps. Review the proposed prose too:
passing those checks does not guarantee every sentence is correct.

In [ ]:
RUN_COMPARISON = MODE == "live" # "rehearsal"   # set True for the live comparison
COMPARISON_CASES = ["INC-001", "INC-004"]
# For all development cases: COMPARISON_CASES = [f"INC-{i:03}" for i in range(1, 7)]

comparison_results = []
if RUN_COMPARISON:
    comparison_results = await run_comparison(COMPARISON_CASES, settings, repeats=1)
    show_summary(comparison_results)
else:
    print("Comparison skipped. Enable when ready, or replay saved live results below.")

### Keep the evidence

Exports include calls, tool traces, grade checks, routing decisions, models, configured prices,
data hashes, and dependency versions. Keys are not exported. Rehearsal and live files are
named separately; the summary rejects mixed-mode results.

In [ ]:
saved_path = None
if comparison_results:
    saved_path = save_results(comparison_results, settings, ROOT / "runs")
    print("Saved:", saved_path.name)

## 5 · Replay recorded live results on stage

Use a JSON produced by a real comparison. This cell makes no calls. The banner explicitly
states that the results were recorded; a rehearsal file keeps its rehearsal label.

In [ ]:
REPLAY_PATH = ""   # example: "runs/live-20261009T120000000000Z.json"
if REPLAY_PATH:
    replay = load_results(ROOT / REPLAY_PATH)
    display(HTML("<h3>Recorded results · no new model calls</h3>"))
    show_summary(replay)
    show_case(next((r for r in replay if r["case_id"] == DEMO_CASE and r["arm"] == "switchyard"), replay[0]))
else:
    print("Set REPLAY_PATH after saving a live comparison.")

## 6 · Evaluate before the event

There are **six incident templates**, with one development case and five evaluation variants
per template. The 30 evaluation cases change service/region labels while preserving each
causal pattern. They are suitable for demonstrating the workflow, not estimating general
production performance. Avoid tuning after looking at evaluation outcomes.

Thirty cases × three arms = **90 agent tasks**, plus all classifier calls. Do this before the
event, after the smaller comparison is stable. Repeat runs help expose variance but require
increasing `max_suite_tasks` intentionally. Do not remove failures from the final comparison.

In [ ]:
RUN_EVALUATION = False
evaluation_results = []
if RUN_EVALUATION:
    evaluation_ids = dataset_table().query("split == 'evaluation'")["id"].tolist()
    evaluation_results = await run_comparison(evaluation_ids, settings, repeats=1, seed=123)
    show_summary(evaluation_results)
    evaluation_path = save_results(evaluation_results, settings, ROOT / "runs")
    print("Saved:", evaluation_path.name)
else:
    print("Evaluation is off. Enable it after checking the development comparison.")

## 7 · What can we claim?

**Presenter, 5:15–7:00:** Report the measured cost, task pass rate, and latency together.
If routing costs more or misses cases, show it. A small/cheap model alone is a real baseline
and may be the best choice for this workload.

- A legitimate claim: “On these synthetic cases, with these models and rates, we measured
  this cost and pass rate; judge calls and failures are included.”
- This does not establish production savings, equal quality, a cold-cache comparison, or a
  reproduction of another benchmark. The price calculation excludes hosting/tool costs and
  provider-specific surcharges. Missing accounting stays unavailable.
- Stage routing and task classification are different algorithms. Changing the router or
  model pair means running the comparison again.
- Live cost is computed from returned usage and configured prices, not verified against an invoice.

**Before presenting:** inspect each model's proposed resolution, keep a recorded live comparison,
and rehearse one straightforward and one misleading-symptom case.

References: [Switchyard](https://github.com/NVIDIA-NeMo/Switchyard) ·
[LangChain integration](https://github.com/langchain-ai/langchain-nvidia/tree/48b296bd37211ff750e95066ef23210b58f55500/libs/switchyard) ·
[Published benchmark](https://www.langchain.com/blog/switchyard-agent-routing-benchmark) ·
[Existing LangChain sample](https://github.com/langchain-samples/smart-model-routing)